In [12]:
import scMPRAforge as scm
from dask_jobqueue import SLURMCluster
from dask.distributed import Client, LocalCluster

In [13]:
local=False
if local:
    cluster=LocalCluster(memory_limit='48G')
    client = Client(cluster)
else:
    cluster=SLURMCluster(
        cores=8,#cores per slurm job
        memory="32G",#memory per slurm job
        processes=1,#dask workers per slurm job
        job_extra_directives=["-p day", 
            f"--job-name=simclust_worker",
            f"--time=3:00:00",
            f"--output=worker_%j.out"]
    )
    cluster.scale(jobs=3)
    client = Client(cluster,
            timeout=f"{10*60}s",   # Client <-> scheduler timeout 
            heartbeat_interval="20s",  # Worker heartbeat interval,
        )

In [14]:
from pathlib import Path

In [15]:
# in the real version, de_novo_sim will take a pair, path/name on init and never save it.
# relative paths to individual components can be used, saved, assumed. 
DATA_ROOT=Path("/home/mcn26/project_pi_skr2/shared/tabula_data")
path=DATA_ROOT/"simulated/shendure_pow_analysis"
name="sim_with_orthos_20251119"

In [16]:
from dask.distributed import Semaphore, as_completed, get_client

In [17]:
ortho_root=path/name/"orthos"
scmpradat_root=path/name/"scMPRA"
output_root=path/name/"orthos_with_precomputed_wald"
output_root.mkdir(exist_ok=True)

input_ortho_names=[path.name for path in ortho_root.iterdir()]

Semaphore(max_leases=3, name="wald-precompute")

def precompute_one_wald(input_root, scmpradat_root, name, output_root):
    sem = Semaphore(name="wald-precompute")
    with sem:
        client=get_client()
        dat=scm.scMPRA_data.from_parquet(scmpradat_root/Path(name).with_suffix(".scmpra"))
        dat.ortho_filter()
        ortho_oi=scm.ortho.load(client=client,
                                path=input_root,
                                name=name)
        ortho_oi.training_data=dat
        ortho_oi.precompute_wald(client)
        ortho_oi.save(path=output_root,name=name)

futures = [client.submit(precompute_one_wald, input_root=ortho_root,scmpradat_root=scmpradat_root,name=name_oi,output_root=output_root) for name_oi in input_ortho_names]

In [18]:
for i in futures:
    i.result()
    print("1x done")

1x done
1x done
1x done
1x done
1x done


In [19]:
client.close()
cluster.close()